In [1]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings('ignore')

In [10]:
import numpy as np
import torch
import torch.nn as nn
from PyPDF2 import PdfReader 
from pdfplumber import pdf
import sys

In [23]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, util
import time
from numba import jit, cuda 

In [16]:
def progressBar(count_value, total, suffix=''):
    bar_length = 100
    filled_up_Length = int(round(bar_length* count_value / float(total)))
    percentage = round(100.0 * count_value/float(total),1)
    bar = '=' * filled_up_Length + '-' * (bar_length - filled_up_Length)
    sys.stdout.write('[%s] %s%s ...%s\r' %(bar, percentage, '%', suffix))
    sys.stdout.flush()


def load_split_pdf(pdf_path):
    pdf_loader = PdfReader(open(pdf_path, "rb"))
    pdf_text = ""
    for page_num in range(len(pdf_loader.pages)):
        pdf_page = pdf_loader.pages[page_num]
        pdf_text += pdf_page.extract_text()
    # progressBar(7, 7)
    return pdf_text

In [17]:
pdf_path = '/home/hkaman/Documents/multimodel-transformers-vye/test.pdf'
pdf_text = load_split_pdf(pdf_path)
# print(len(pdf_text))

In [24]:
def split_text_using_RCTS(pdf_text):
    text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2048,
    chunk_overlap=64
    )
    split_texts = text_splitter.split_text(pdf_text)
    paragraphs = []
    for text in split_texts:
        paragraphs.extend(text.split('\n')) 
    # progressBar(3, 7)
    return paragraphs

In [27]:
pdf_text_split = split_text_using_RCTS(pdf_text)

In [47]:
def Initialize_sentence_transformer():
    model_name = "clip-ViT-L-14" #"sentence-transformers/all-MiniLM-L6-v2" , "sentence-transformers/nq-distilbert-base-v1"
    embeddings = SentenceTransformer(model_name)
    # progressBar(4, 7)
    return embeddings

def encode_each_paragraph(paragraphs, embeddings):
    responses = []
    for paragraph in paragraphs:
        response = embeddings.encode([paragraph], convert_to_tensor=True)
        responses.append((paragraph, response))
    # progressBar(5, 7)
    return responses

In [48]:
embeddings = Initialize_sentence_transformer()

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

0_CLIPModel/config.json:   0%|          | 0.00/4.54k [00:00<?, ?B/s]

0_CLIPModel/merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

...

0_CLIPModel/special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

0_CLIPModel/tokenizer_config.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

0_CLIPModel/preprocessor_config.json:   0%|          | 0.00/354 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

0_CLIPModel/vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

In [49]:
model = SentenceTransformer('clip-ViT-L-14')
text_emb = model.encode(pdf_text_split)

In [51]:
text_emb.shape

(205, 768)

In [45]:
tensors = encode_each_paragraph(pdf_text_split, embeddings)
len(tensors)

205

In [46]:
# Extract the tensors from the list of tuples
tensor_list = [t[1] for t in tensors]

# Concatenate the tensors along dimension 0
concatenated_tensors = torch.cat(tensor_list, dim=0)
concatenated_tensors.shape

torch.Size([205, 768])

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertModel

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
tokens = tokenizer.encode('This is a input.', return_tensors='pt')
print("These are tokens!", tokens)
for token in tokens[0]:
    print("This are decoded tokens!", tokenizer.decode([token]))

model = DistilBertModel.from_pretrained("distilbert-base-uncased")
print(model.embeddings.word_embeddings(tokens))
for e in model.embeddings.word_embeddings(tokens)[0]:
    print("This is an embedding!", e)

In [17]:
emb1 = model.embeddings.word_embeddings(tokens1)
emb1.shape

torch.Size([1, 25, 768])

In [18]:
emb2 = model.embeddings.word_embeddings(tokens2)

In [4]:
text_sample_1 = 'Chardonnay vines on 4WIREW0 trellises, 10m row spacing, 6m canopy.'
text_sample_2 = 'Chardonnay vines on 4WIREW0 trellises, 10m row spacing, 6m canopy.'

In [5]:
tokens1 = tokenizer.encode(text_sample_1, return_tensors='pt')
tokens2 = tokenizer.encode(text_sample_2, return_tensors='pt')

In [7]:
tokens2

tensor([[  101, 25869,  5280, 16741, 16702,  2006,  1018, 20357,  2860,  2692,
         29461, 21711,  2229,  1010,  2184,  2213,  5216, 12403,  6129,  1010,
          1020,  2213, 14582,  1012,   102]])

In [6]:
tokens1

tensor([[  101, 25869,  5280, 16741, 16702,  2006,  1018, 20357,  2860,  2692,
         29461, 21711,  2229,  1010,  2184,  2213,  5216, 12403,  6129,  1010,
          1020,  2213, 14582,  1012,   102]])

In [ ]:
text_tensor_embedings = nn.Embedding(64, 512)
emb_tensor1 = text_tensor_embedings(tokens1)
emb_tensor1

In [8]:
for token in tokens1[0]:
    print("This are decoded tokens!", tokenizer.decode([token]))

This are decoded tokens! [CLS]
This are decoded tokens! char
This are decoded tokens! ##don
This are decoded tokens! ##nay
This are decoded tokens! vines
This are decoded tokens! on
This are decoded tokens! 4
This are decoded tokens! ##wire
This are decoded tokens! ##w
This are decoded tokens! ##0
This are decoded tokens! tre
This are decoded tokens! ##llis
This are decoded tokens! ##es
This are decoded tokens! ,
This are decoded tokens! 10
This are decoded tokens! ##m
This are decoded tokens! row
This are decoded tokens! spa
This are decoded tokens! ##cing
This are decoded tokens! ,
This are decoded tokens! 6
This are decoded tokens! ##m
This are decoded tokens! canopy
This are decoded tokens! .
This are decoded tokens! [SEP]
